# 数据类与枚举

学习目标：用数据类和枚举表达结构明确的数据对象，并说明默认值、相等比较和可变性的设计选择。

前置知识：类与继承、装饰器、类型标注、容器与对象引用、异常处理。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

## 1 字段与自动生成的方法

### 1.1 用字段描述一条记录

数据类（data class）用带类型标注的字段描述记录。dataclasses.dataclass 是类装饰器，会按字段定义顺序生成初始化、表示和比较方法。下面用 StockItem 表示一种物品及其数量，quantity 默认为 0。

这里沿用类、装饰器和类型标注；本章侧重数据对象的字段和行为选择，不重复展开这些机制。

| dataclass 参数 | 中文名称／含义 | 默认行为 |
| --- | --- | --- |
| init | 生成初始化方法 | True：生成 \_\_init\_\_ |
| repr | 生成对象表示方法 | True：生成 \_\_repr\_\_，显示类名与字段 |
| eq | 生成相等比较方法 | True：生成 \_\_eq\_\_，按参与比较的字段判断相等 |
| order | 生成顺序比较方法 | False：不生成小于、大于等顺序比较方法 |

默认的相等比较要求两个实例类型相同，再比较字段值；值相等不代表是同一个对象。类若已自行定义 \_\_init\_\_、\_\_repr\_\_ 或 \_\_eq\_\_，相应方法不会被覆盖。dataclasses.fields() 可取得字段描述，下面读取其中的 name。

In [1]:
import dataclasses


@dataclasses.dataclass
class StockItem:
    """记录一种物品及其当前数量。"""

    name: str
    quantity: int = 0


item = StockItem("铅笔")
same_item = StockItem(name="铅笔", quantity=0)
print(item)  # StockItem(name='铅笔', quantity=0)
print(item == same_item, item is same_item)  # True False
print([field.name for field in dataclasses.fields(StockItem)])
# ['name', 'quantity']：顺序来自字段定义。

StockItem(name='铅笔', quantity=0)
True False
['name', 'quantity']


### 1.2 标注不会自动校验输入

dataclass 使用标注识别字段，但不会按普通字段的标注检查或转换实参。下面故意传入字符串数量，创建对象仍会成功。

需要限制输入类型或数值范围时，必须显式校验；第 4 节演示初始化后的业务校验。

In [2]:
unchecked_item = StockItem("铅笔", quantity="5")
print(unchecked_item.quantity, type(unchecked_item.quantity).__name__)
# 5 str：保留了字符串，没有自动转成整数。
assert unchecked_item.quantity == "5"

5 str


## 2 默认值与 default_factory

### 2.1 每个实例独立创建可变默认值

列表等可变字段使用 dataclasses.field(default_factory=list)。default_factory 接收一个不需要参数的可调用对象；缺少该字段实参时才调用它取得默认值。这里传入 list 本身，不是 list() 的结果。

本例中，每次调用 list 都创建新列表，所以两个默认列表互不影响。显式传入同一列表仍会共享它，工厂不会替调用者复制实参。default 和 default_factory 不能同时指定。

In [3]:
@dataclasses.dataclass
class Checklist:
    """记录检查单名称及其标签。"""

    title: str
    labels: list[str] = dataclasses.field(default_factory=list)


first_checklist = Checklist("读书")
second_checklist = Checklist("练习")
first_checklist.labels.append("已读")
print(first_checklist.labels, second_checklist.labels)  # ['已读'] []
assert first_checklist.labels is not second_checklist.labels

shared_labels = ["待办"]
left = Checklist("任务 A", labels=shared_labels)
right = Checklist("任务 B", labels=shared_labels)
print(left.labels is right.labels)  # True：显式实参仍是同一列表。

['已读'] []
True


### 2.2 不可哈希默认值会在装饰时被拒绝

Python 3.12 的 dataclass 会拒绝不可哈希的字段默认值，并抛出 ValueError。直接把空列表写成默认值就会触发这个检查，错误发生在类被装饰时，而不是创建实例时。

这是对常见可变默认值问题的防护；可哈希与不可变并不等价，不能据此认定所有共享默认对象都安全。

In [4]:
try:
    @dataclasses.dataclass
    class InvalidChecklist:
        """反例：直接把列表设为字段默认值。"""

        labels: list[str] = []
except ValueError as error:
    print(type(error).__name__)  # ValueError：改用 default_factory=list。
else:
    raise AssertionError("不可哈希的默认列表应被拒绝")

ValueError


## 3 选择参与显示、相等与排序的字段

field(repr=False) 从生成的对象表示中排除字段；field(compare=False) 从生成的相等和顺序比较中排除字段。两者互相独立，都不阻止直接读取该属性。

本例约定：队列项先按 priority 排序，同优先级再按 title 排序；note 只是备注，不影响这两个队列项是否相等。priority 的整数越小，排得越靠前。

order=True 会生成 \_\_lt\_\_、\_\_le\_\_、\_\_gt\_\_、\_\_ge\_\_，按参与比较的字段顺序进行元组式比较，且要求实例类型相同。它要求 eq=True；若类已有这些顺序比较方法之一，再设置 order=True 会抛出 TypeError。

In [5]:
@dataclasses.dataclass(order=True)
class QueueEntry:
    """按优先级和标题比较队列项，忽略备注。"""

    priority: int
    title: str
    note: str = dataclasses.field(default="", repr=False, compare=False)


entry = QueueEntry(1, "B", note="已联系")
print(entry)  # QueueEntry(priority=1, title='B')
print(entry.note)  # 已联系：仍可直接读取。
print(entry == QueueEntry(1, "B", note="待联系"))  # True

entries = [QueueEntry(2, "A"), entry, QueueEntry(1, "A")]
print([(entry.priority, entry.title) for entry in sorted(entries)])
# [(1, 'A'), (1, 'B'), (2, 'A')]：先比较 priority，再比较 title。

QueueEntry(priority=1, title='B')
已联系
True
[(1, 'A'), (1, 'B'), (2, 'A')]


## 4 初始化后的校验与派生字段

生成的 \_\_init\_\_ 完成字段赋值后，会调用类中定义的 \_\_post_init\_\_。可以在这里校验字段之间的关系，或计算依赖其他字段的值。若没有生成初始化方法，这个钩子不会自动调用。

field(init=False) 把字段从生成的初始化参数中排除。下面的 count 表示含首尾的页数，由 start 和 stop 计算，不由调用者填写。

本例约定输入页码是整数，只校验起始页至少为 1、结束页不早于起始页。这不是完整的运行时类型校验；钩子也不会在后续属性赋值时重新执行。

In [6]:
@dataclasses.dataclass
class PageRange:
    """记录含首尾的页码区间，并计算创建时的页数。"""

    start: int
    stop: int
    count: int = dataclasses.field(init=False)

    def __post_init__(self) -> None:
        """检查页码关系并计算页数。"""
        if self.start < 1 or self.stop < self.start:
            raise ValueError("页码须满足 1 <= start <= stop")
        self.count = self.stop - self.start + 1


pages = PageRange(3, 5)
print(pages)  # PageRange(start=3, stop=5, count=3)

try:
    PageRange(5, 3)
except ValueError as error:
    print(str(error))  # 页码须满足 1 <= start <= stop
else:
    raise AssertionError("倒置的页码区间应被拒绝")

PageRange(start=3, stop=5, count=3)
页码须满足 1 <= start <= stop


## 5 frozen 与哈希边界

### 5.1 冻结属性不等于冻结内部对象

frozen=True 使普通的属性赋值和删除抛出 dataclasses.FrozenInstanceError，从而模拟只读实例。它不会递归冻结字段引用的对象。

下面的 labels 仍是列表，append 会修改该列表；给 labels 换一个列表则是给数据类属性赋值，会被拒绝。

In [7]:
@dataclasses.dataclass(frozen=True)
class FrozenChecklist:
    """冻结属性绑定，但保留列表字段以观察内部可变性。"""

    title: str
    labels: list[str] = dataclasses.field(default_factory=list)


frozen_checklist = FrozenChecklist("发布")
frozen_checklist.labels.append("已检查")
print(frozen_checklist.labels)  # ['已检查']：内部列表仍可修改。

try:
    frozen_checklist.labels = []
except dataclasses.FrozenInstanceError as error:
    print(type(error).__name__)  # FrozenInstanceError
else:
    raise AssertionError("冻结实例不应允许重新绑定 labels")

['已检查']
FrozenInstanceError


### 5.2 生成哈希方法仍需字段支持

保持默认的 eq=True、unsafe_hash=False，且没有自定义哈希方法时，普通数据类不可哈希；加上 frozen=True 则会生成 \_\_hash\_\_。生成哈希方法不保证调用成功，参与哈希的字段也必须可哈希，内部列表会使 hash() 抛出 TypeError。

若设置 eq=False，dataclass 保留原有的哈希行为，可能继承基类实现。不要为了让可变对象进入集合就设置 unsafe_hash=True：对象作为键期间的哈希值必须保持稳定，相等对象也必须具有相同哈希值。

下面用字符串和整数构成文档键；只检查等值对象的哈希是否相等，不依赖某个具体哈希数值。

In [8]:
@dataclasses.dataclass(frozen=True)
class DocumentKey:
    """用标题和版本号表示可作为字典键的文档值。"""

    title: str
    edition: int = 1


document_key = DocumentKey("指南", 2)
same_key = DocumentKey("指南", 2)
locations = {document_key: "shelf-A"}
print(locations[same_key])  # shelf-A：等值键可查到同一项。
assert hash(document_key) == hash(same_key)

# 分别检查：普通数据类不可哈希；冻结实例的列表字段也不可哈希。
for record in (StockItem("铅笔"), frozen_checklist):
    try:
        hash(record)
    except TypeError:
        print(type(record).__name__, "不可哈希")
    else:
        raise AssertionError("这两个示例都应无法计算哈希")

shelf-A
StockItem 不可哈希
FrozenChecklist 不可哈希


## 6 继承中的字段与参数顺序

### 6.1 合并基类字段与子类字段

数据类会合并数据类基类的字段，再加入自身字段；子类可覆盖同名字段的类型或默认值。下面只使用单继承，覆盖 edition 的默认值后，它仍保留在原有字段位置，新增的 pages 排在最后。

子类生成的初始化方法会负责这些字段的赋值。本例的基类也由 dataclass 生成初始化方法，无需再手工调用它。

In [9]:
@dataclasses.dataclass
class Document:
    """记录文档标题和版本号。"""

    title: str
    edition: int = 1


@dataclasses.dataclass
class TextDocument(Document):
    """在文档字段之外增加页数，并采用新的默认版本号。"""

    pages: int = 0
    edition: int = 2


document = TextDocument("指南", pages=12)
print(document)  # TextDocument(title='指南', edition=2, pages=12)
print([field.name for field in dataclasses.fields(TextDocument)])
# ['title', 'edition', 'pages']：覆盖字段不移到最后。
print(Document("指南", 2) == TextDocument("指南", 2))
# False：字段有重叠，但不是同一种数据类。

TextDocument(title='指南', edition=2, pages=12)
['title', 'edition', 'pages']
False


### 6.2 必填字段不能排在普通默认参数之后

生成的初始化方法仍遵循参数顺序要求：普通参数中，无默认值的字段不能排在有默认值的字段之后。检查的是继承后合并的顺序，因此下面新增的必填 pages 会因基类 edition 有默认值而触发 TypeError。

若 pages 应当必填，可以用 field(kw_only=True) 把它设为仅限关键字参数。这样调用者必须写 pages=12，它不会进入普通位置参数的顺序约束。

In [10]:
try:
    @dataclasses.dataclass
    class InvalidDocument(Document):
        """反例：继承默认字段之后增加普通必填字段。"""

        pages: int
except TypeError as error:
    print(type(error).__name__)  # TypeError：合并后默认参数在前。
else:
    raise AssertionError("默认字段之后的普通必填字段应被拒绝")


@dataclasses.dataclass
class RequiredPagesDocument(Document):
    """要求调用者通过关键字明确给出页数。"""

    pages: int = dataclasses.field(kw_only=True)


print(RequiredPagesDocument("指南", pages=12))
# RequiredPagesDocument(title='指南', edition=1, pages=12)

TypeError
RequiredPagesDocument(title='指南', edition=1, pages=12)


## 7 用 Enum 表达有限状态

### 7.1 成员、名称、值与遍历

枚举（enumeration）把一组符号名称绑定到值，适合表达有限的选项。继承 enum.Enum 定义枚举类，成员名通常用大写字母。下面的 JobState 成员分别表示排队、运行中和完成。

成员的 name 是名称，value 是对应值。JobState["QUEUED"] 按名称查找，JobState("queued") 按值查找；两者得到同一个已存在成员。遍历枚举按定义顺序得到成员。

In [11]:
import enum


class JobState(enum.Enum):
    """表示任务的三个状态。"""

    QUEUED = "queued"
    RUNNING = "running"
    DONE = "done"


state = JobState.QUEUED
print(state.name, state.value)  # QUEUED queued
print(JobState["QUEUED"] is state, JobState("queued") is state)
# True True：查找返回已定义的成员。
print([member.name for member in JobState])
# ['QUEUED', 'RUNNING', 'DONE']

QUEUED queued
True True
['QUEUED', 'RUNNING', 'DONE']


### 7.2 查找错误与普通枚举的比较

对这里定义的 JobState，找不到名称会抛出 KeyError，找不到值会抛出 ValueError。名称和字符串值可能采用不同拼写，不能混用两种查找形式。

普通 Enum 的成员可以比较相等，但不会仅因 value 相同就等于字符串或整数，也不自动支持小于、大于比较。状态的排列顺序不等于业务上的先后约束。

In [12]:
print(JobState.QUEUED == "queued")  # False：成员不是它的字符串值。

try:
    JobState["queued"]
except KeyError as error:
    print(type(error).__name__)  # KeyError：这里需要名称 QUEUED。
else:
    raise AssertionError("不存在的小写成员名称应查找失败")

try:
    JobState("paused")
except ValueError as error:
    print(type(error).__name__)  # ValueError：没有值为 paused 的成员。
else:
    raise AssertionError("未知状态值应被拒绝")

try:
    JobState.QUEUED < JobState.RUNNING
except TypeError as error:
    print(type(error).__name__)  # TypeError：普通 Enum 不提供大小关系。
else:
    raise AssertionError("普通枚举不应自动提供顺序比较")

False
KeyError
ValueError
TypeError


### 7.3 别名不会新增独立成员

同一枚举允许多个名称使用相同值。后定义的名称成为先定义成员的别名（alias），按名称或值查找都会得到那个成员，因此别名的 name 也是最先定义的名称。

正常遍历跳过别名；\_\_members\_\_ 是名称到成员的只读有序映射，包含别名。下面用 WAITING 作为旧状态名称的兼容入口。

In [13]:
class LegacyState(enum.Enum):
    """保留 WAITING 作为 QUEUED 的兼容别名。"""

    QUEUED = "queued"
    WAITING = "queued"
    DONE = "done"


print(LegacyState.WAITING is LegacyState.QUEUED)  # True
print(LegacyState.WAITING.name)  # QUEUED
print([member.name for member in LegacyState])  # ['QUEUED', 'DONE']
print(list(LegacyState.__members__))  # ['QUEUED', 'WAITING', 'DONE']

True


QUEUED
['QUEUED', 'DONE']
['QUEUED', 'WAITING', 'DONE']


### 7.4 用 unique 拒绝别名

如果每个名称都应代表一个独立值，使用 enum.unique 装饰器。它在类创建后检查是否有别名，发现重复值便抛出 ValueError。

这与枚举本身禁止重复定义同一个名称不同：unique 检查的是不同名称是否指向同一个成员。

In [14]:
try:
    @enum.unique
    class InvalidState(enum.Enum):
        """反例：要求唯一值，却重复使用 queued。"""

        QUEUED = "queued"
        WAITING = "queued"
except ValueError as error:
    print(type(error).__name__)  # ValueError：发现 WAITING 别名。
else:
    raise AssertionError("unique 应拒绝重复值")

ValueError


### 7.5 需要兼容整数或字符串时

| 类型 | 中文名称／含义 | 与原始值的关系 |
| --- | --- | --- |
| Enum | 普通枚举 | 成员不会仅因值相同而等于原始值 |
| IntEnum | 整数枚举 | 成员也是整数，可以参与整数运算和比较 |
| StrEnum | 字符串枚举 | 成员也是字符串，可以参与字符串操作和比较 |

IntEnum、StrEnum 适合兼容已有整数或字符串常量；这种兼容也弱化了成员与原始值的区分。整数或字符串运算的结果会离开枚举类型。StrEnum 从 Python 3.11 提供，本章的 3.12 可直接使用。

个别接口要求精确的 str 类型，而不接受子类，此时可显式调用 str()。如果只需明确区分有限状态，本章的数据对象仍采用普通 Enum。

In [15]:
class Priority(enum.IntEnum):
    """兼容已有的整数优先级。"""

    HIGH = 1
    LOW = 2


class FileKind(enum.StrEnum):
    """兼容已有的文件类型字符串。"""

    TEXT = "txt"
    CSV = "csv"


print(Priority.HIGH == 1, FileKind.TEXT == "txt")  # True True
print(type(Priority.HIGH + 1).__name__)  # int：计算结果不是 Priority。
print(type(FileKind.TEXT.upper()).__name__)  # str：结果不是 FileKind。
print(type(str(FileKind.TEXT)) is str)  # True：转换为普通字符串。

True True
int
str
True


## 8 设计一个任务数据对象

本例约定：任务允许更新状态，采用普通数据类；title 和 state 共同决定值是否相等，处理备注 notes 不参与相等比较。每个任务的默认备注列表独立创建。

有限状态由 JobState 表达，输入处先把字符串值转换成成员，再创建 Job。state 的标注本身仍不会检查或转换输入，也不会验证状态能否从排队直接跳到完成。

这里只在初始化时检查标题不是空白；实际业务若要求每次修改都维护约束，需要设计相应的修改入口。示例聚焦字段设计，不加入文件读写或服务层。

In [16]:
@dataclasses.dataclass
class Job:
    """保存可更新状态的任务，把处理备注排除在值比较之外。"""

    title: str
    state: JobState = JobState.QUEUED
    notes: list[str] = dataclasses.field(
        default_factory=list,
        repr=False,
        compare=False,
    )

    def __post_init__(self) -> None:
        """拒绝只有空白字符的任务标题。"""
        if not self.title.strip():
            raise ValueError("任务标题不能为空白")


incoming = {"title": "生成目录", "state": "queued"}
job = Job(incoming["title"], state=JobState(incoming["state"]))
job.notes.append("输入已检查")
job.state = JobState.RUNNING

print(job.state.name, job.notes)  # RUNNING ['输入已检查']
print(job == Job("生成目录", JobState.RUNNING))  # True：备注不影响相等。
print(Job("另一任务").notes)  # []：备注列表不共享。

try:
    Job("   ")
except ValueError as error:
    print(str(error))  # 任务标题不能为空白
else:
    raise AssertionError("空白标题应在初始化时被拒绝")

RUNNING ['输入已检查']
True
[]
任务标题不能为空白


## 本章小结

（1）dataclass 根据字段生成方法；普通类型标注不自动校验或转换输入，相等比较要求实例类型相同。

（2）可变默认值用 default_factory 创建；显式传入同一对象仍可能共享。repr 与 compare 分别决定字段是否参与对象表示和比较。

（3）初始化后的钩子可校验关系、计算派生字段；frozen 只模拟属性只读，哈希还取决于字段类型和稳定性。

（4）继承会合并字段；默认值约束需要看合并后的参数顺序，必填字段可按需要设为仅限关键字参数。

（5）Enum 表达有限选项，名称和值使用不同查找形式；别名不增加独立成员，unique 可拒绝别名。

自查：能否解释一个对象为什么相等、哪些内容允许修改，以及它是否适合作为字典键？

## 练习

（1）先预测下方三次输出，再运行核对。解释：修改默认列表为什么只影响一个检查单，列表内容是否参与相等比较，以及别名为何影响名称观察却不增加遍历项。

In [17]:
exercise_first = Checklist("读书")
exercise_second = Checklist("读书")
exercise_first.labels.append("已读")

print(exercise_first == exercise_second)
print(LegacyState.WAITING.name)
print([member.name for member in LegacyState])
# 先写下预测，再展开输出核对；从字段比较和别名两条规则解释。

False


QUEUED
['QUEUED', 'DONE']


（2）实现冻结的数据类 BookKey，字段为 isbn 和 edition，edition 默认为 1。在初始化后的钩子中拒绝小于 1 的版本号，本题按标注传入字符串和整数。

检查标准：相同 ISBN 和版本号的两个对象相等，放入集合后只有一项；不同版本号不相等；修改 edition 抛出 FrozenInstanceError；版本号为 0 抛出 ValueError。只比较等值对象的哈希是否相等，不固定哈希数值。

In [18]:
book_arguments = {"isbn": "9787111000001", "edition": 2}

# 在这里定义 BookKey，并用 book_arguments 构造两个等值对象。
# 检查集合长度、不同版本号、冻结赋值和非法版本号。
# 反例只捕获预期异常，若未抛出则用 AssertionError 指出失败。

（3）定义带 unique 的 ExportFormat 枚举，TEXT 的值为 "txt"，CSV 的值为 "csv"。再定义继承本章 Document 的数据类 ExportRequest，增加没有默认值的 format 字段，并把它设为仅限关键字参数。

使用下方输入先解析枚举值，再创建请求。检查标准：保留 Document 的 edition 默认值 1；相同标题、版本、格式的请求相等，格式不同则不相等；不传 format 时抛出 TypeError；未知格式值抛出 ValueError。解释为什么不能把 format 直接写成普通必填字段。

In [19]:
export_input = {"title": "学习记录", "format": "csv"}
unknown_format = "xml"

# 在这里定义 ExportFormat 和 ExportRequest。
# 构造时用 ExportFormat 解析输入，并通过 format= 传入枚举成员。
# 分别检查默认版本、字段比较、缺少格式和未知格式。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [dataclass：字段识别、默认值、生成方法与哈希规则](https://docs.python.org/3.12/library/dataclasses.html#dataclasses.dataclass)；[field：default_factory、init、repr、compare、kw_only](https://docs.python.org/3.12/library/dataclasses.html#dataclasses.field)；[fields：字段描述](https://docs.python.org/3.12/library/dataclasses.html#dataclasses.fields)；[默认工厂](https://docs.python.org/3.12/library/dataclasses.html#default-factory-functions)、[可变默认值](https://docs.python.org/3.12/library/dataclasses.html#mutable-default-values)、[初始化后处理](https://docs.python.org/3.12/library/dataclasses.html#post-init-processing)、[冻结实例](https://docs.python.org/3.12/library/dataclasses.html#frozen-instances)、[继承](https://docs.python.org/3.12/library/dataclasses.html#inheritance)、[仅限关键字参数的重排](https://docs.python.org/3.12/library/dataclasses.html#re-ordering-of-keyword-only-parameters-in-init)；[对象引用与可变性](https://docs.python.org/3.12/reference/datamodel.html#objects-values-and-types)、[哈希约束](https://docs.python.org/3.12/reference/datamodel.html#object.__hash__)；[Enum HOWTO：有限选项与成员](https://docs.python.org/3.12/howto/enum.html#enum-howto)、[按名称和值访问](https://docs.python.org/3.12/howto/enum.html#programmatic-access-to-enumeration-members-and-their-attributes)、[名称重复与值别名](https://docs.python.org/3.12/howto/enum.html#duplicating-enum-members-and-values)、[遍历与完整成员映射](https://docs.python.org/3.12/howto/enum.html#iteration)、[枚举比较](https://docs.python.org/3.12/howto/enum.html#comparisons)；[名称查找与 KeyError](https://docs.python.org/3.12/library/enum.html#enum.EnumType.__getitem__)、[按定义顺序遍历](https://docs.python.org/3.12/library/enum.html#enum.EnumType.__iter__)、[unique](https://docs.python.org/3.12/library/enum.html#enum.unique)、[IntEnum](https://docs.python.org/3.12/library/enum.html#enum.IntEnum)、[StrEnum](https://docs.python.org/3.12/library/enum.html#enum.StrEnum)。 |
| CPython 官方源码（GitHub，v3.12.14） | [Lib/enum.py，第 1158–1178 行](https://github.com/python/cpython/blob/v3.12.14/Lib/enum.py#L1158-L1178)：普通 Enum 未找到对应值且未定制缺失值处理时，抛出 ValueError；用于状态解析和未知格式反例。 |